## Prepare data
Load a multirep session from a `session.pkl` file, session directory, or `.tar.gz` archive.

In [36]:
import sys, importlib.util
from pathlib import Path

# VS Code injects __vsc_ipynb_file__ with the notebook's absolute path.
_nb = globals().get("__vsc_ipynb_file__") or ""
if not _nb:
    raise RuntimeError("Select the project venv kernel: .venv/bin/python")

REPO_ROOT = Path(_nb).resolve().parent.parent  # analysis/ -> repo root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Import directly — skips analysis/__init__.py (avoids eager matplotlib/torch load)
def _load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_loader = _load_module("multirep_loader", REPO_ROOT / "analysis" / "multirep_loader.py")
load_session              = _loader.load_session
load_session_from_tarball = _loader.load_session_from_tarball

# Point to a session directory, session.pkl, or .tar.gz archive
SESSION_PATH = REPO_ROOT / "experiment" / "data" / "multirepData" / "fast-test-local_20260603_092104.tar.gz"

path = Path(SESSION_PATH)
if path.suffixes[-2:] == [".tar", ".gz"]:
    session = load_session_from_tarball(path)
else:
    session = load_session(path)

print(f"Session:    {session.session_id}")
print(f"Preset:     {session.preset_name}")
print(f"Timestamp:  {session.session_timestamp}")
print(f"Tasks:      {session.n_tasks}")
print(f"Datasets:   {session.datasets}")
print(f"Rep rows:   {len(session.reputation_timeline)}")
print(f"Acc rows:   {len(session.global_accuracy)}")

Session:    ca9d6b3c-8e24-4065-968e-dfbac76c4fc6
Preset:     fast-test-local
Timestamp:  20260603_092104
Tasks:      10
Datasets:   ['mnist', 'cifar-10', 'mnist', 'cifar-10', 'mnist', 'cifar-10', 'mnist', 'cifar-10', 'mnist', 'cifar-10']
Rep rows:   120
Acc rows:   40


## Access top-level data

In [37]:
rep     = session.reputation_timeline   # one row per (task, user)
acc     = session.global_accuracy       # one row per (task, round)
tasks   = session.tasks                 # list of per-task metadata dicts
preset  = session.preset

## Reputation timeline
One row per `(task_index, user)`. Contains pre/post reputation values, selection decision, and confidence.

**Key columns:** `task_index`, `dataset`, `task_type`, `user_name`, `behavior`, `was_selected`, `was_cached`, `tr_pre`, `tr_post`, `tr_all_post`, `gir_pre`, `gir_post`, `q_pre`, `q_post`, `balance_pre`, `balance_post`, `selection_score`, `confidence`, `k`

In [38]:
rep

,task_index,dataset,task_type,fingerprint,user_name,user_address,guid,behavior,was_selected,was_cached,...,tr_all_post,gir_post,q_post,q_all_post,balance_post,total_contrib_post,confidence,k,running_c_mean,m2
0,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,Balanced A,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,296baa1b-ecd2-e910-43b2-ed0f556c8caf,honest,True,False,...,{5: 0.0},0.000000,0.416667,{5: 0.4166666666666667},0.0,0.0,0.000000,0,0.000000,0.000000
1,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,Balanced B,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,607f92d4-7776-41d1-9fd8-d1544b28d35e,honest,True,False,...,{5: 0.0},0.000000,0.416667,{5: 0.4166666666666667},0.0,0.0,0.000000,0,0.000000,0.000000
2,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,MNIST Expert 1,0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC,bee79562-79c0-d807-1571-44d0b752438e,honest,True,False,...,{5: 0.0},0.000000,0.416667,{5: 0.4166666666666667},0.0,0.0,0.000000,0,0.000000,0.000000
3,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,MNIST Expert 2,0x90F79bf6EB2c4f870365E785982E1f101E93b906,ac193986-b823-1eb7-af58-2839a4c10301,honest,False,False,...,{5: 0.0},0.005556,0.000000,{5: 0.0},0.0,0.0,0.166667,1,0.000000,0.000000
4,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,CIFAR Expert 1,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,627ce64d-fccf-0d4b-3c15-a6105eefa35a,honest,False,False,...,{5: 0.0},0.000000,0.000000,{5: 0.0},0.0,0.0,0.166667,1,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,9,cifar-10,6,24a9fb229b762f4c755111e913e3748b404fa46f05381c...,Cross Mal-MNIST Honest-CIFAR,0x14dC79964da2C08b23698B3D3cc7Ca32193d9955,b7fd4edd-361d-6858-0a0e-37ce12be516c,malicious,False,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.068056,0.166667,"{5: 0.16666666666666666, 1: 0.0, 2: 0.0, 3: 0....",0.0,0.0,0.139555,2,0.484788,0.052366
116,9,cifar-10,6,24a9fb229b762f4c755111e913e3748b404fa46f05381c...,Malicious Both,0x23618e81E3f5cdF7f54C3d65f7FBc0aBf5B21E8f,31c4d68b-1f18-071e-dd3b-499b715548f9,malicious,False,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.068056,0.166667,"{5: 0.16666666666666666, 1: 0.0, 2: 0.0, 3: 0....",0.0,0.0,0.074359,2,0.188492,0.142117
117,9,cifar-10,6,24a9fb229b762f4c755111e913e3748b404fa46f05381c...,Free-rider Both,0xa0Ee7A142d267C1f36714E4a8F75612F20a79720,34eac3e3-3c1c-9ae8-841a-94fcb2cf42e5,freerider,False,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.129600,0.833333,"{5: 0.75, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0...",0.0,0.0,0.155415,3,0.308449,0.070645
118,9,cifar-10,6,24a9fb229b762f4c755111e913e3748b404fa46f05381c...,Passive A,0xBcd4042DE499D14e55001CcbB24a551F3b954096,520d0604-83ca-5769-ec11-2bcb5483c0b2,honest,False,False,...,"{5: 0.004377104377104377, 1: 0.0, 2: 0.0, 3: 0...",0.124722,0.333333,"{5: 0.4166666666666667, 1: 0.0, 2: 0.0, 3: 0.0...",0.0,0.0,0.184453,2,0.675270,0.027449


## Global accuracy
One row per `(task_index, round)`. Aggregated from per-task run data.

**Key columns:** `task_index`, `dataset`, `round`, `objective_global_accuracy`, `objective_global_loss`, `reward_pool`, `punishment_pool`

In [39]:
acc

,round,round_time,objective_global_accuracy,objective_global_loss,reward_pool,punishment_pool,task_index,dataset
0,0,0.000000,0.0844,2.302734,1000000000000000000,0,0,mnist
1,1,1.900683,0.4478,2.263672,666666666666666667,666666666666666664,0,mnist
2,2,1.719468,0.6138,2.214844,333333333333333334,333333333333333333,0,mnist
3,3,1.709037,0.6863,1.985352,1,818181818181818183,0,mnist
4,0,0.000000,0.0996,2.306641,1000000000000000000,0,1,cifar-10
5,1,2.513965,0.1000,2.302734,666666666666666667,0,1,cifar-10
6,2,2.508629,0.1000,2.302734,333333333333333334,0,1,cifar-10
7,3,1.478698,0.1049,2.302734,1,0,1,cifar-10
8,0,0.000000,0.0986,2.302734,1000000000000000000,0,2,mnist
9,1,3.785873,0.5937,2.173828,666666666666666667,666666666666666664,2,mnist


## Per-task run data
Each task in `session.tasks` may embed the full tables from its individual `.pkl` run.
Use `session.get_task_run_data(task_index)` to retrieve them, or iterate with `session.iter_run_data()`.

**Tables per task:** `global`, `users`, `votes`, `receipts`, `contributions`, `warnings`, `task_rep_calc`, `trs`

In [40]:
import pandas as pd

rows = []
for t in tasks:
    rd = t.get("run_data")
    has_data = rd is not None and any(
        (hasattr(df, "empty") and not df.empty) for df in rd.values()
    )
    rows.append({
        "task_index": t["task_index"],
        "dataset":    t["dataset"],
        "task_type":  t["task_type"],
        "was_cached": t["was_cached"],
        "has_run_data": has_data,
    })

pd.DataFrame(rows)

,task_index,dataset,task_type,was_cached,has_run_data
0,0,mnist,5,False,True
1,1,cifar-10,6,False,True
2,2,mnist,5,False,True
3,3,cifar-10,6,False,True
4,4,mnist,5,False,True
5,5,cifar-10,6,False,True
6,6,mnist,5,False,True
7,7,cifar-10,6,False,True
8,8,mnist,5,False,True
9,9,cifar-10,6,False,True


In [41]:
# Inspect a specific task
TASK_INDEX = 1

rd = session.get_task_run_data(TASK_INDEX)
if rd is None:
    print(f"Task {TASK_INDEX} has no embedded run data.")
else:
    global_df       = rd.get("global",        pd.DataFrame())
    users_df        = rd.get("users",          pd.DataFrame())
    votes_df        = rd.get("votes",          pd.DataFrame())
    receipts_df     = rd.get("receipts",       pd.DataFrame())
    contributions_df= rd.get("contributions",  pd.DataFrame())
    task_rep_df     = rd.get("task_rep_calc",  pd.DataFrame())
    warnings_df     = rd.get("warnings",       pd.DataFrame())
    trs_df          = rd.get("trs",            pd.DataFrame())

    for name, df in [("global", global_df), ("users", users_df), ("votes", votes_df),
                     ("receipts", receipts_df), ("contributions", contributions_df),
                     ("task_rep_calc", task_rep_df), ("warnings", warnings_df), ("trs", trs_df)]:
        print(f"{name}: {df.shape}")

global: (4, 7)
users: (20, 17)
votes: (60, 11)
receipts: (68, 5)
contributions: (15, 17)
task_rep_calc: (5, 13)
warnings: (0, 0)
trs: (1, 5)


In [42]:
global_df

,experiment_id,round,round_time,objective_global_accuracy,objective_global_loss,reward_pool,punishment_pool
0,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,0,0.000000,0.0996,2.306641,1000000000000000000,0
1,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,2.513965,0.1000,2.302734,666666666666666667,0
2,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,2,2.508629,0.1000,2.302734,333333333333333334,0
3,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,3,1.478698,0.1049,2.302734,1,0


In [43]:
users_df

,experiment_id,round,user_number,state,behavior,role,grs,address,id,subjective_personal_accuracy,subjective_personal_loss,subjective_global_accuracy,subjective_global_loss,round_reputation_assigned,reward_delta,is_reward,merged
0,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,0,5,active,Attitude.Honest,Attitude.Honest,1000000000000000000,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,4acd829e-785a-446e-b566-83093e782195,NaN,NaN,NaN,NaN,NaN,NaN,None,None
1,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,0,4,active,Attitude.Honest,Attitude.Honest,1000000000000000000,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,de79e16c-379b-4df9-892d-ca1d92051514,NaN,NaN,NaN,NaN,NaN,NaN,None,None
2,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,0,11,active,Attitude.Honest,Attitude.Honest,1000000000000000000,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,3ace7a9e-37a3-45bf-b6df-cde2ad349139,NaN,NaN,NaN,NaN,NaN,NaN,None,None
3,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,0,0,active,Attitude.Honest,Attitude.Honest,1000000000000000000,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,28c040fd-6b3f-4bd7-9d0b-b72e6fba0ed4,NaN,NaN,NaN,NaN,NaN,NaN,None,None
4,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,0,10,active,Attitude.Honest,Attitude.Honest,1000000000000000000,0xBcd4042DE499D14e55001CcbB24a551F3b954096,5982d327-116f-44da-9d75-8034337b7cb8,NaN,NaN,NaN,NaN,NaN,NaN,None,None
5,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,5,active,Attitude.Honest,Attitude.Honest,1065217391304347659,None,4acd829e-785a-446e-b566-83093e782195,0.0000,2.294922,0.1017,2.302734,2.000000e+18,6.521739e+16,True,True
6,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,4,active,Attitude.Honest,Attitude.Honest,1065217391304347659,None,de79e16c-379b-4df9-892d-ca1d92051514,0.4000,2.236328,0.0999,2.302734,4.000000e+18,6.521739e+16,True,True
7,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,11,active,Attitude.Honest,Attitude.Honest,1086956521739130225,None,3ace7a9e-37a3-45bf-b6df-cde2ad349139,0.0900,2.301563,0.1022,2.300781,2.000000e+18,8.695652e+16,True,True
8,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,0,active,Attitude.Honest,Attitude.Honest,1086956521739130225,None,28c040fd-6b3f-4bd7-9d0b-b72e6fba0ed4,0.1475,2.297188,0.1442,2.296875,2.000000e+18,8.695652e+16,True,True
9,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,10,active,Attitude.Honest,Attitude.Honest,1028985507246377563,None,5982d327-116f-44da-9d75-8034337b7cb8,0.0900,2.301797,0.0957,2.302734,2.000000e+18,2.898551e+16,True,True


In [44]:
votes_df

,experiment_id,round,giver_id,receiver_id,giver_address,receiver_address,vote_feedback_score,vote_prev_accuracy,vote_prev_loss,vote_accuracy,vote_loss
0,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,4acd829e-785a-446e-b566-83093e782195,de79e16c-379b-4df9-892d-ca1d92051514,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,1,NaN,232,333,230
1,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,4acd829e-785a-446e-b566-83093e782195,3ace7a9e-37a3-45bf-b6df-cde2ad349139,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,1,NaN,232,5000,227
2,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,4acd829e-785a-446e-b566-83093e782195,28c040fd-6b3f-4bd7-9d0b-b72e6fba0ed4,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,1,NaN,232,2000,227
3,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,4acd829e-785a-446e-b566-83093e782195,5982d327-116f-44da-9d75-8034337b7cb8,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,0xBcd4042DE499D14e55001CcbB24a551F3b954096,1,NaN,232,0,235
4,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,de79e16c-379b-4df9-892d-ca1d92051514,4acd829e-785a-446e-b566-83093e782195,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,-1,333.0,222,0,231
5,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,de79e16c-379b-4df9-892d-ca1d92051514,3ace7a9e-37a3-45bf-b6df-cde2ad349139,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,-1,333.0,222,0,234
6,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,de79e16c-379b-4df9-892d-ca1d92051514,28c040fd-6b3f-4bd7-9d0b-b72e6fba0ed4,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,-1,333.0,222,0,234
7,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,de79e16c-379b-4df9-892d-ca1d92051514,5982d327-116f-44da-9d75-8034337b7cb8,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,0xBcd4042DE499D14e55001CcbB24a551F3b954096,-1,333.0,222,0,232
8,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,3ace7a9e-37a3-45bf-b6df-cde2ad349139,4acd829e-785a-446e-b566-83093e782195,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,1,900.0,231,900,230
9,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,3ace7a9e-37a3-45bf-b6df-cde2ad349139,de79e16c-379b-4df9-892d-ca1d92051514,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,1,900.0,231,1200,230


In [45]:
receipts_df

,experiment_id,round,tx_type,tx_hash,gas_used
0,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,slot,40727ac6b69d883be3e0270c917e1391cad3221317d176...,50837
1,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,slot,36f514604e4f9c571b20bbf23e3ea16fa21224b2866e90...,50825
2,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,slot,022daaa248f9c610cff3b2606a4dad323c0f335f7b3bcc...,50825
3,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,slot,dc89e0ec9067f108ae1b261b36436e1fc61d580f19ddb9...,50837
4,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,slot,85dea38152068cc1e00b81ca5fa8be9cb500f4cf5f2f0e...,50837
...,...,...,...,...,...
63,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,4,exit,7b7fa2eac573e09f22aa44b13af1e56de31a039bb2eb88...,47476
64,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,4,exit,e0587db8ae8312455617522656f51c6d38fd0abcb20351...,50040
65,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,4,exit,9396a00fb794d9cabee79d228583b72877ddc05693898a...,52604
66,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,4,exit,a18849b4f1f34953cfa94441bb401bc8bdbd4f62339451...,55168


In [46]:
contributions_df

,experiment_id,round,user_id,user_address,contribution_score,user_mad_avg,current_excluded_values,current_accepted_values,current_mad_median,current_mad_value,current_mad_max_deviation,prev_avg_round_val_after_mad,previous_excluded_values,previous_accepted_values,previous_mad_median,previous_mad_value,previous_mad_max_deviation
0,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,5,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,195652173913042987,230.250000,[],"[231, 230, 230, 230]",230.0,0.0,NaN,231.0,[222],"[232, 231, 230, 231]",231.0,1.0,1.630838
1,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,4,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,195652173913042987,230.250000,[],"[230, 230, 230, 231]",230.0,0.0,NaN,231.0,[222],"[232, 231, 230, 231]",231.0,1.0,1.630838
2,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,11,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,260869565217390686,230.000000,"[227, 234]","[230, 230]",230.0,1.5,2.446256,231.0,[222],"[232, 231, 230, 231]",231.0,1.0,1.630838
3,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,0,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,260869565217390686,230.000000,"[227, 234]","[230, 230]",230.0,1.5,2.446256,231.0,[222],"[232, 231, 230, 231]",231.0,1.0,1.630838
4,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,1,10,0xBcd4042DE499D14e55001CcbB24a551F3b954096,86956521739132694,230.666667,[235],"[232, 230, 230]",231.0,1.0,1.630838,231.0,[222],"[232, 231, 230, 231]",231.0,1.0,1.630838
5,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,2,5,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,649999999999991473,230.250000,[],"[231, 230, 230, 230]",230.0,0.0,NaN,230.4,[],"[231, 231, 230, 230, 230]",230.0,0.0,NaN
6,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,2,4,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,649999999999991473,230.250000,[],"[231, 230, 230, 230]",230.0,0.0,NaN,230.4,[],"[231, 231, 230, 230, 230]",230.0,0.0,NaN
7,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,2,11,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,-99999999999994315,230.500000,[],"[231, 231, 230, 230]",230.5,0.5,0.815419,230.4,[],"[231, 231, 230, 230, 230]",230.0,0.0,NaN
8,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,2,0,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,-99999999999994315,230.500000,[],"[231, 231, 230, 230]",230.5,0.5,0.815419,230.4,[],"[231, 231, 230, 230, 230]",230.0,0.0,NaN
9,cifar-10-loss_only-1-0.1-1-1.0-False-{e4504c3a...,2,10,0xBcd4042DE499D14e55001CcbB24a551F3b954096,-99999999999994315,230.500000,[],"[231, 231, 230, 230]",230.5,0.5,0.815419,230.4,[],"[231, 231, 230, 230, 230]",230.0,0.0,NaN


In [47]:
import pandas as pd

all_contributions = pd.concat([
    rd.get("contributions", pd.DataFrame()).assign(task_index=task_index)
    for task_index, dataset, rd in session.iter_run_data()
    if rd.get("contributions") is not None and not rd["contributions"].empty
], ignore_index=True)

all_contributions["contribution_score"] /= 1e18

user_names = rep[["user_address", "user_name"]].drop_duplicates()
all_contributions = all_contributions.merge(user_names, on="user_address", how="left")
all_contributions["user"] = all_contributions["user_name"].fillna(all_contributions["user_id"].astype(str))

pivot = all_contributions.pivot_table(
    index="user",
    columns="task_index",
    values="contribution_score",
    aggfunc="mean"
).rename_axis(None, axis=1)

selected_counts = all_contributions.groupby("task_index")["user"].nunique().rename("# selected")
pivot.loc["# selected"] = selected_counts
pivot

/tmp/ipykernel_24799/509570012.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_contributions = pd.concat([


,0,1,2,3,4,5,6,7,8,9
user,,,,,,,,,,
Balanced A,NaN,0.120290,NaN,NaN,1.021959,NaN,0.425,NaN,1.0,0.250000
Balanced B,NaN,NaN,0.723717,NaN,1.231694,NaN,1.050,NaN,NaN,NaN
CIFAR Expert 1,NaN,0.348551,NaN,-0.292484,NaN,NaN,NaN,NaN,NaN,NaN
CIFAR Expert 2,NaN,0.348551,NaN,-0.059641,NaN,NaN,NaN,NaN,NaN,NaN
Cross Honest-MNIST Mal-CIFAR,0.939394,NaN,0.414424,NaN,0.869500,-0.033333,NaN,NaN,NaN,0.250000
Cross Mal-MNIST Honest-CIFAR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,0.291666
Free-rider Both,NaN,NaN,NaN,-0.833333,-1.000000,0.633333,NaN,NaN,NaN,NaN
MNIST Expert 1,NaN,NaN,NaN,-0.526144,-0.869500,0.355556,-1.000,NaN,NaN,NaN
MNIST Expert 2,NaN,NaN,NaN,NaN,NaN,0.077778,NaN,NaN,NaN,NaN


In [ ]:
task_rep_df

## Merge users with their votes (per task)
Attach receiver behavior/role to each vote row.

In [ ]:
if votes_df.empty or users_df.empty:
    print("votes or users is empty for this task.")
else:
    votes_with_receiver = votes_df.merge(
        users_df[["experiment_id", "round", "address", "behavior", "role"]],
        left_on=["experiment_id", "round", "receiver_address"],
        right_on=["experiment_id", "round", "address"],
        how="left"
    ).rename(columns={
        "behavior": "receiver_behavior",
        "role":     "receiver_role",
    }).drop(columns=["address"])

    votes_with_receiver

## Reputation analysis across all tasks

In [ ]:
# Selection frequency per user
rep[rep["was_selected"]].groupby(["user_name", "behavior"])["task_index"].count().rename("times_selected").reset_index()

In [ ]:
# Mean final TR per behavior group
rep.groupby("behavior")[["tr_post", "gir_post", "q_post", "balance_post"]].mean().round(4)

In [ ]:
# Filter reputation timeline to a specific task
rep[rep["task_index"] == TASK_INDEX]

## Jupyter notes

Display all columns:

```python
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
```

Iterate all tasks with run data:

```python
for task_index, dataset, rd in session.iter_run_data():
    global_df = rd.get("global", pd.DataFrame())
    ...
```